In [1]:
# 03_feature_engineering.ipynb

import pandas as pd
import numpy as np
import os
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder
import joblib

# --- Setup Paths ---
DATA_PATH = "../data"
MODELS_PATH = "../models"
os.makedirs(MODELS_PATH, exist_ok=True)

target_col = "isFraud"

In [2]:
# 1. Load cleaned merged data (Path Correction Applied)
train = pd.read_csv(os.path.join(DATA_PATH, "train_merged_clean.csv"))
test  = pd.read_csv(os.path.join(DATA_PATH, "test_merged_clean.csv"))

print("Train:", train.shape, " Test:", test.shape)

Train: (590540, 360)  Test: (506691, 378)


In [3]:
# 2. Add time features from TransactionDT
def add_time_features(df):
    if "TransactionDT" not in df.columns:
        return df
    
    # Time delta is in seconds
    df["TransactionDT_days"] = df["TransactionDT"] / (24*60*60)
    df["Transaction_day"] = np.floor(df["TransactionDT_days"])
    df["Transaction_hour"] = (df["TransactionDT"] // 3600) % 24
    df["Transaction_weekday"] = df["Transaction_day"] % 7 # Assuming day 0 is Monday or Sunday
    return df

train = add_time_features(train)
test  = add_time_features(test)

In [4]:
# 3. Log transform TransactionAmt
if "TransactionAmt" in train.columns:
    for df in [train, test]:
        df["TransactionAmt_log"] = np.log1p(df["TransactionAmt"])

In [5]:
        
# 4. Save TransactionID and target separately
train_ids = train["TransactionID"]
test_ids  = test["TransactionID"]

y = train[target_col]

In [6]:
# 5. Identify dtypes and align features (FIX APPLIED HERE)
# Get the shared columns between train and test, excluding the target column
shared_cols = list(set(train.columns) & set(test.columns))
feature_cols = [c for c in shared_cols if c not in [target_col, "TransactionID"]]

# Select and copy features
train_features = train[feature_cols].copy()
test_features  = test[feature_cols].copy() 

# Separate numeric and string/object columns
numeric_cols = train_features.select_dtypes(include=["int64", "float64"]).columns.tolist()
object_cols = train_features.select_dtypes(include=["object"]).columns.tolist() 

print("Numeric cols :", len(numeric_cols))
print("Object cols  :", len(object_cols))

Numeric cols : 328
Object cols  : 16


In [7]:
# 6. Imputers
NUMERIC_IMPUTE_VALUE = -999 
CAT_IMPUTE_VALUE = "missing"

num_imputer = SimpleImputer(strategy="constant", fill_value=NUMERIC_IMPUTE_VALUE)
cat_imputer = SimpleImputer(strategy="constant", fill_value=CAT_IMPUTE_VALUE)

# --- Apply Imputation ---\
# Fit on train, apply to both train & test
train_features[numeric_cols] = num_imputer.fit_transform(train_features[numeric_cols])
test_features[numeric_cols]  = num_imputer.transform(test_features[numeric_cols])

if object_cols:
    train_features[object_cols] = cat_imputer.fit_transform(train_features[object_cols])
    test_features[object_cols]  = cat_imputer.transform(test_features[object_cols])
    
# --- New Function for Target/Mean Encoding with Smoothing ---\
def get_target_encoding(train_df, test_df, col, target_col, alpha=50):
    """
    Computes smoothed target encoding using a global mean.
    
    alpha (smoothing): higher alpha means more smoothing for low-count categories.
    """
    global_mean = train_df[target_col].mean()
    
    # 1. Calculate Mean and Count for each category in the training set
    agg = train_df.groupby(col)[target_col].agg(['mean', 'count'])
    
    # 2. Apply smoothing formula: (count * mean + alpha * global_mean) / (count + alpha)
    smoothing_factor = agg['count'] / (agg['count'] + alpha)
    encoding = global_mean * (1 - smoothing_factor) + agg['mean'] * smoothing_factor
    
    # 3. Apply encoding to the features (using the mapping from the training set)
    train_encoded = train_df[col].map(encoding)
    test_encoded = test_df[col].map(encoding)
    
    # Fill NaN (categories unseen in train) with the global mean
    train_encoded.fillna(global_mean, inplace=True)
    test_encoded.fillna(global_mean, inplace=True)
    
    return train_encoded, test_encoded, encoding

# --- New Function for Frequency Encoding ---\
def get_frequency_encoding(train_df, test_df, col):
    """Computes the frequency of each category."""
    counts = train_df[col].value_counts(normalize=True).to_dict()
    
    train_encoded = train_df[col].map(counts)
    test_encoded = test_df[col].map(counts)
    
    # Fill NaN (categories unseen in train) with 0
    test_encoded.fillna(0, inplace=True)
    
    return train_encoded, test_encoded, counts

In [8]:
# 7. Apply Specialized Encoding for Categoricals
target_encoded_cols = []
frequency_encoded_cols = []
encoding_mappings = {}
cols_to_drop = []

# Target Encode key predictive features (from EDA)
for col in ['ProductCD', 'card4', 'card6', 'DeviceType', 'P_emaildomain']:
    new_col = f'{col}_target_enc'
    if col in object_cols:
        
        # Temporarily create a copy of the features + target for encoding fit
        train_temp = train_features.copy()
        train_temp[target_col] = y
        
        train_enc, test_enc, mapping = get_target_encoding(
            train_temp, test_features, col, target_col, alpha=50
        )
        
        train_features[new_col] = train_enc
        test_features[new_col] = test_enc
        target_encoded_cols.append(new_col)
        encoding_mappings[new_col] = mapping
        
        cols_to_drop.append(col) # Mark original column for dropping

# Remove target encoded columns from the object_cols list to avoid re-encoding
object_cols = [col for col in object_cols if col not in cols_to_drop]

# Frequency Encode the rest of the object columns
for col in object_cols:
    new_col = f'{col}_freq_enc'
    
    train_enc, test_enc, mapping = get_frequency_encoding(
        train_features, test_features, col
    )
    
    train_features[new_col] = train_enc
    test_features[new_col] = test_enc
    frequency_encoded_cols.append(new_col)
    encoding_mappings[new_col] = mapping
    
    cols_to_drop.append(col) # Mark original column for dropping

# Drop all original object columns
train_features.drop(columns=cols_to_drop, inplace=True)
test_features.drop(columns=cols_to_drop, inplace=True)
  

C:\Users\hp\AppData\Local\Temp\ipykernel_5720\1530272766.py:20: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  train_features[new_col] = train_enc
C:\Users\hp\AppData\Local\Temp\ipykernel_5720\1530272766.py:21: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  test_features[new_col] = test_enc
C:\Users\hp\AppData\Local\Temp\ipykernel_5720\1530272766.py:20: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at o

In [9]:
  
# 8. Final Feature List Check (All columns should now be numeric)
final_numeric_cols = train_features.columns.tolist()
print(f"\nTotal final features (all numeric): {len(final_numeric_cols)}")


Total final features (all numeric): 344


In [10]:
# 9. Rebuild final train/test dataframes
train_final = train_features.copy()
train_final[target_col] = y.values
train_final["TransactionID"] = train_ids.values

test_final = test_features.copy()
test_final["TransactionID"] = test_ids.values

print("Final train shape:", train_final.shape)
print("Final test shape :", test_final.shape)

Final train shape: (590540, 346)
Final test shape : (506691, 345)


In [11]:
# 10. Save final processed datasets
train_final.to_csv(os.path.join(DATA_PATH, "train_final_processed.csv"), index=False)
test_final.to_csv(os.path.join(DATA_PATH, "test_final_processed.csv"), index=False)

In [12]:
# 11. Save preprocessing objects for deployment / reuse
preprocessing_bundle = {
    "numeric_cols": numeric_cols,
    "target_encoded_cols": target_encoded_cols,
    "frequency_encoded_cols": frequency_encoded_cols,
    "num_imputer": num_imputer,
    "cat_imputer": cat_imputer,
    "encoding_mappings": encoding_mappings,
    "feature_cols": final_numeric_cols,
    "NUMERIC_IMPUTE_VALUE": NUMERIC_IMPUTE_VALUE, # save constant for reference
}

joblib.dump(preprocessing_bundle, os.path.join(MODELS_PATH, "preprocessing_bundle.pkl"))

print(f"\nSaved train_final_processed.csv, test_final_processed.csv and preprocessing_bundle.pkl to {DATA_PATH} and {MODELS_PATH}")


Saved train_final_processed.csv, test_final_processed.csv and preprocessing_bundle.pkl to ../data and ../models
